In [24]:

import pandas as pd

df = pd.read_csv("laptop_price.csv", encoding="latin1")

df.head()



,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


In [25]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

df["Ram"] = df["Ram"].str.replace("GB", "").astype(float)
df["Weight"] = df["Weight"].str.replace("kg", "").astype(float)

features = ["Company", "TypeName", "Inches", "Ram", "Weight",
            "Cpu", "Memory", "Gpu", "OpSys"]

X = df[features]
y = df["Price_euros"]

prep = ColumnTransformer([
    ("num", StandardScaler(), ["Inches", "Ram", "Weight"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     ["Company", "TypeName", "Cpu", "Memory", "Gpu", "OpSys"])
])

model = make_pipeline(prep, Lasso(alpha=0.1, max_iter=10000))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Laptop Model Ready! 💻")
print("Error (€):", round(mean_absolute_error(y_test, predictions), 2))



Laptop Model Ready! 💻
Error (€): 206.1


In [26]:
new_laptop = pd.DataFrame({
    "Company": ["Dell"],
    "TypeName": ["Gaming"],
    "Inches": [15.6],
    "Ram": [16.0],
    "Weight": [2.5],
    "Cpu": ["Intel Core i7 12700H 2.3GHz"],
    "Memory": ["512GB SSD"],
    "Gpu": ["Nvidia GeForce RTX 3060"],
    "OpSys": ["Windows 11"]
})

print("Predicted Price (€):", round(model.predict(new_laptop)[0], 2))

Predicted Price (€): 1350.06
